# Lab 5: Building ETL Pipelines using PySpark
## Course: Data Stores & Pipelines (Trimester 2)

Welcome to Lab 5! In this worked example, we will design and execute an end-to-end **ETL (Extract, Transform, Load)** pipeline using **PySpark**. 

### Learning Objectives:
* **Extract:** Initialize a `SparkSession` and ingest raw datasets.
* **Transform:** Clean data, handle missing values, apply business logic, and perform structural modifications.
* **Load:** Write the processed dataframe to target storage formats efficiently.

### Step 1: Initialize SparkSession
Every PySpark application begins with creating a `SparkSession`, which serves as the entry point to programming Spark with the Dataset and DataFrame API.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, when, lit, current_timestamp

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("Lab5_PySpark_ETL_Pipeline") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

### Step 2: Extract Phase
In the Extract phase, we ingest data from our source system. For this lab, we will simulate raw data ingestion by creating a sample DataFrame representing raw transaction records.

In [ ]:
# Simulate Extract: Creating raw data
raw_data = [
    (1, "alice", "electronics", " 1200.50 ", "2026-03-01"),
    (2, "bob", "clothing", " 45.00 ", "2026-03-02"),
    (3, "charlie", "electronics", None, "2026-03-02"),
    (4, "david", "home", " 150.00 ", "2026-03-03"),
    (5, "eve", "clothing", " 89.99 ", "2026-03-04"),
    (6, "frank", "electronics", " 350.00 ", None)
]

columns = ["transaction_id", "customer_name", "category", "amount_str", "date_str"]

raw_df = spark.createDataFrame(raw_data, columns)
print("--- Raw Extracted Data ---")
raw_df.show(truncate=False)

### Step 3: Transform Phase
The Transform phase is where we clean, validate, and restructure our data. Here we will:
1. Cast string amounts to numeric double types.
2. Handle missing or null values.
3. Standardize text columns (e.g., uppercase names and categories).
4. Add an audit column (`processed_at`) for tracking.

In [ ]:
# Transform: Cleaning and enriching data
from pyspark.sql.types import DoubleType

transformed_df = raw_df \
    .withColumn("amount", col("amount_str").cast(DoubleType())) \
    .na.fill({"amount": 0.0, "date_str": "1970-01-01"}) \
    .withColumn("category", upper(col("category"))) \
    .withColumn("customer_name", upper(col("customer_name"))) \
    .withColumn("processed_at", current_timestamp()) \
    .drop("amount_str")

print("--- Transformed Data ---")
transformed_df.show(truncate=False)

### Step 4: Load Phase
In the Load phase, we write our transformed DataFrame into a target data store or file format (e.g., Parquet, Delta Lake, or data warehouse tables).

In [ ]:
# Load: Writing output to Parquet format (simulated destination)
output_path = "./output/transformed_transactions"

transformed_df.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"ETL Pipeline completed successfully! Data loaded to: {output_path}")